# Data Inventory and Quality Audit

This notebook establishes whether the current jobs snapshot is fit for specific downstream studies. It inventories the data before any market metric is produced, keeps every denominator visible, and separates observed readiness from validated outcomes.

## Scope and non-goals

**In scope:** the current local CSV export of the Supabase `jobs` snapshot, covering approximately 4.5 months.

**Not in scope:** annual or year-over-year claims, S3 reconciliation, production writes, destructive deduplication, salary normalization, or claims that `state` is geographically correct. This notebook reads one local file and writes nothing.

## 1. Reproducibility contract

The file name, checksum, dimensions, and cutoff are fixed review inputs. Execution stops with an actionable error if the local snapshot differs. The cutoff is derived from the snapshot contract rather than the machine clock.

In [ ]:
from __future__ import annotations

import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
SNAPSHOT_RELATIVE_PATH = Path("data/analysis/jobs_27_07_2026.csv")
EXPECTED_SHA256 = "c3412e3315ee4c3df02f9a4f9c0e570747518f9e1ad70f9a6e1e86ceb260fd71"
EXPECTED_ROWS = 26_619
EXPECTED_COLUMNS = 26
SNAPSHOT_CUTOFF = pd.Timestamp("2026-07-27 23:59:59.999999999", tz="UTC")
EXPECTED_DATE_START = pd.Timestamp("2026-03-11", tz="UTC")
EXPECTED_DATE_END = pd.Timestamp("2026-07-27", tz="UTC")
EXPECTED_SCHEMA = [
    "id", "site", "job_url", "title", "company", "raw_location",
    "country", "state", "career", "scraped_by", "industry_niche",
    "is_premium", "job_type", "salary_min", "salary_max", "currency",
    "salary_source", "description", "cleaned_description", "keywords_json",
    "embedding", "date_posted", "date_scraped", "last_updated",
    "status", "country_code",
]
STRING_COLUMNS = [column for column in EXPECTED_SCHEMA if column not in {
    "salary_min", "salary_max", "is_premium"
}]
NA_TOKENS = ["", "NULL", "null"]

In [ ]:
def locate_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook or repository working directory.

    Args:
        start: Directory from which to search upward.

    Returns:
        Repository root containing both pyproject.toml and the expected data parent.

    Raises:
        FileNotFoundError: If no matching repository root is found.
    """
    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Repository root not found from {resolved_start}. "
        "Run this notebook from inside the bajio-industrial-scout-pipeline repository."
    )


def sha256_file(path: Path, chunk_size: int = 1_048_576) -> str:
    """Compute a file SHA-256 digest without loading the whole file into memory.

    Args:
        path: File to hash.
        chunk_size: Number of bytes read per chunk.

    Returns:
        Lowercase hexadecimal SHA-256 digest.
    """
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
repo_root = locate_repo_root(Path.cwd())
snapshot_path = repo_root / SNAPSHOT_RELATIVE_PATH
if not snapshot_path.is_file():
    raise FileNotFoundError(
        f"Expected ignored snapshot at {snapshot_path}. "
        "Place the validated export there before execution."
    )

actual_sha256 = sha256_file(snapshot_path)
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(
        "Snapshot checksum mismatch. "
        f"Expected {EXPECTED_SHA256}, received {actual_sha256}. "
        "Stop and verify the export before analysis."
    )

pd.Series({
    "snapshot": str(SNAPSHOT_RELATIVE_PATH),
    "sha256": actual_sha256,
    "expected_rows": EXPECTED_ROWS,
    "expected_columns": EXPECTED_COLUMNS,
    "fixed_cutoff_utc": SNAPSHOT_CUTOFF.isoformat(),
}, name="reproducibility_contract")

## 2. Load and schema inventory

The standard CSV parser honors quoted multiline descriptions. Identifier, URL, and text-like fields use pandas' nullable string dtype so leading characters are preserved and missing values have one representation.

In [ ]:
jobs = pd.read_csv(
    snapshot_path,
    encoding="utf-8",
    sep=",",
        quotechar='"',
    dtype={column: "string" for column in STRING_COLUMNS},
    keep_default_na=True,
    na_values=NA_TOKENS,
    on_bad_lines="error",
    low_memory=False,
)
if jobs.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(
        f"Snapshot dimensions mismatch: expected {(EXPECTED_ROWS, EXPECTED_COLUMNS)}, "
        f"received {jobs.shape}. Stop before analysis."
    )

jobs.shape

In [ ]:
actual_columns = jobs.columns.tolist()
missing_columns = [column for column in EXPECTED_SCHEMA if column not in actual_columns]
unexpected_columns = [column for column in actual_columns if column not in EXPECTED_SCHEMA]
schema_inventory = pd.DataFrame({
    "metric": [
        "expected columns", "actual columns", "missing expected columns",
        "unexpected columns", "deep memory MiB",
    ],
    "value": [
        len(EXPECTED_SCHEMA), len(actual_columns), len(missing_columns),
        len(unexpected_columns), round(jobs.memory_usage(deep=True).sum() / 2**20, 2),
    ],
    "detail": [
        EXPECTED_SCHEMA, actual_columns, missing_columns, unexpected_columns,
        "DataFrame memory after loading; not CSV file size",
    ],
})
schema_inventory

In [ ]:
pd.DataFrame({
    "column": jobs.columns,
    "dtype": jobs.dtypes.astype(str).to_numpy(),
    "non_null_rows": jobs.notna().sum().to_numpy(),
    "memory_mib": (jobs.memory_usage(deep=True, index=False) / 2**20).round(3).to_numpy(),
})

## 3. Key integrity

Stable IDs and canonical job URLs define which duplicate studies are possible. Counts distinguish duplicate rows from excess copies; no rows are removed here.

In [ ]:
def nonempty_text(series: pd.Series) -> pd.Series:
    """Return a Boolean mask for non-null, non-whitespace text values."""
    return series.astype("string").str.strip().fillna("").ne("")


key_rows = []
for key in ["id", "job_url"]:
    populated = nonempty_text(jobs[key])
    duplicate_rows = jobs.loc[populated, key].duplicated(keep=False).sum()
    excess_copies = jobs.loc[populated, key].duplicated(keep="first").sum()
    key_rows.append({
        "key": key,
        "empty_rows": int((~populated).sum()),
        "duplicate_rows_involved": int(duplicate_rows),
        "excess_duplicate_copies": int(excess_copies),
        "denominator_rows": len(jobs),
    })
key_integrity = pd.DataFrame(key_rows)
key_integrity

## 4. Temporal coverage

Parsing failures, gaps, and future timestamps can bias time-based summaries. Coverage is measured against the fixed July 27 snapshot cutoff, never `now()`. Days covered is inclusive of both endpoints.

In [ ]:
analysis_jobs = jobs.assign(
    date_scraped_parsed=pd.to_datetime(jobs["date_scraped"], errors="coerce", utc=True),
    last_updated_parsed=pd.to_datetime(jobs["last_updated"], errors="coerce", utc=True),
)

temporal_rows = []
for source_column, parsed_column in [
    ("date_scraped", "date_scraped_parsed"),
    ("last_updated", "last_updated_parsed"),
]:
    parsed = analysis_jobs[parsed_column]
    source_present = nonempty_text(analysis_jobs[source_column])
    minimum = parsed.min()
    maximum = parsed.max()
    temporal_rows.append({
        "field": source_column,
        "parsed_rows": int(parsed.notna().sum()),
        "unparseable_present_rows": int((source_present & parsed.isna()).sum()),
        "minimum_utc": minimum,
        "maximum_utc": maximum,
        "inclusive_days": (maximum.normalize() - minimum.normalize()).days + 1,
        "after_fixed_cutoff_rows": int((parsed > SNAPSHOT_CUTOFF).sum()),
    })
temporal_coverage = pd.DataFrame(temporal_rows)
temporal_coverage

In [ ]:
scrape_dates = analysis_jobs["date_scraped_parsed"].dt.normalize()
observed_daily_counts = scrape_dates.value_counts().sort_index()
calendar = pd.date_range(EXPECTED_DATE_START, EXPECTED_DATE_END, freq="D")
daily_counts = observed_daily_counts.reindex(calendar, fill_value=0).rename("job_count").to_frame()
daily_counts.index.name = "date_utc"
daily_counts["share_of_snapshot_pct"] = (daily_counts["job_count"] / len(jobs) * 100).round(2)
missing_calendar_dates = daily_counts.index[daily_counts["job_count"].eq(0)]

pd.Series({
    "observed_min_date": scrape_dates.min(),
    "observed_max_date": scrape_dates.max(),
    "expected_min_date": EXPECTED_DATE_START,
    "expected_max_date": EXPECTED_DATE_END,
    "missing_calendar_date_count": len(missing_calendar_dates),
    "missing_calendar_dates": [date.date().isoformat() for date in missing_calendar_dates],
}, name="date_scraped_contract")

### Diagnostic: daily job arrivals

Each cell is one calendar day's job count, arranged by week for compact review. Bars are comparable across the matrix, and the caption retains the all-row denominator. This is an arrival diagnostic, not a market trend claim.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily_plot = daily_counts.reset_index()
daily_plot = daily_plot.assign(
    week_start=daily_plot["date_utc"] - pd.to_timedelta(daily_plot["date_utc"].dt.weekday, unit="D"),
    weekday=daily_plot["date_utc"].dt.day_name().str[:3],
)
daily_arrival_matrix = daily_plot.pivot(
    index="week_start", columns="weekday", values="job_count"
).reindex(columns=WEEKDAYS).fillna(0).astype(int)
daily_arrival_matrix.style.bar(
    subset=WEEKDAYS, color="#4C78A8"
).format("{:,}").set_caption(
    f"Daily arrivals by week; denominator = {len(jobs):,} snapshot rows"
)

## 5. Source and workflow coverage

Category mix reveals whether later comparisons are dominated by a source, workflow state, scraper, or collection niche. Missing values remain visible as `<MISSING>`.

In [ ]:
def category_coverage(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Count category values and shares with explicit snapshot denominators.

    Args:
        frame: Source records.
        columns: Categorical columns to summarize.

    Returns:
        Long-form category counts and shares.
    """
    summaries = []
    for column in columns:
        values = frame[column].astype("string").fillna("<MISSING>")
        counts = values.value_counts(dropna=False).rename_axis("value").reset_index(name="count")
        counts.insert(0, "field", column)
        counts["share_pct"] = (counts["count"] / len(frame) * 100).round(2)
        counts["denominator_rows"] = len(frame)
        summaries.append(counts)
    return pd.concat(summaries, ignore_index=True)


workflow_coverage = category_coverage(
    jobs, ["site", "status", "scraped_by", "industry_niche"]
)
workflow_coverage

### Diagnostic: site mix

Bars show counts, while labels retain each site's share and the full-snapshot denominator.

In [ ]:
site_mix = workflow_coverage.loc[
    workflow_coverage["field"].eq("site"),
    ["value", "count", "share_pct", "denominator_rows"],
].set_index("value")
site_mix.style.bar(subset=["count"], color="#F58518").format({
    "count": "{:,}",
    "share_pct": "{:.2f}%",
    "denominator_rows": "{:,}",
}).set_caption("Site mix; every share uses all snapshot rows")

## 6. Critical-field missingness and coverage

Presence is a prerequisite, not proof of semantic quality. Text fields count only non-whitespace values; serialized empty NLP containers such as `[]`, `{}`, and `null` do not count as ready.

In [ ]:
CRITICAL_FIELDS = [
    "description", "cleaned_description", "salary_min", "salary_max",
    "currency", "raw_location", "state", "company", "keywords_json",
    "embedding",
]
SERIALIZED_EMPTY_VALUES = {"[]", "{}", "null", "none"}


def field_present(series: pd.Series, serialized: bool = False) -> pd.Series:
    """Return presence after excluding whitespace and optional empty containers.

    Args:
        series: Values to classify.
        serialized: Whether serialized empty containers should count as missing.

    Returns:
        Boolean presence mask aligned to the source index.
    """
    normalized = series.astype("string").str.strip()
    present = normalized.fillna("").ne("")
    if serialized:
        present &= ~normalized.str.lower().isin(SERIALIZED_EMPTY_VALUES).fillna(False)
    return present


coverage_rows = []
for field in CRITICAL_FIELDS:
    present = field_present(jobs[field], serialized=field in {"keywords_json", "embedding"})
    coverage_rows.append({
        "field": field,
        "present_rows": int(present.sum()),
        "missing_rows": int((~present).sum()),
        "coverage_pct": round(present.mean() * 100, 2),
        "denominator_rows": len(jobs),
    })
critical_coverage = pd.DataFrame(coverage_rows)
critical_coverage

### Diagnostic: critical-field coverage

The bar is percentage coverage; present, missing, and denominator counts prevent percentages from hiding scale.

In [ ]:
critical_coverage.set_index("field").style.bar(
    subset=["coverage_pct"], color="#54A24B", vmin=0, vmax=100
).format({
    "present_rows": "{:,}",
    "missing_rows": "{:,}",
    "coverage_pct": "{:.2f}%",
    "denominator_rows": "{:,}",
}).set_caption("Critical-field presence; denominator = all snapshot rows")

## 7. Salary quality audit

This section tests parseability and range structure only. It does not infer pay periods, convert currencies, normalize values, or make trend claims.

In [ ]:
salary_numeric = jobs[["salary_min", "salary_max"]].apply(pd.to_numeric, errors="coerce")
salary_source_present = jobs[["salary_min", "salary_max"]].notna()
lower_present = salary_numeric["salary_min"].notna()
upper_present = salary_numeric["salary_max"].notna()
paired_bounds = lower_present & upper_present
partial_bounds = lower_present ^ upper_present
any_bound = lower_present | upper_present
inverted_ranges = paired_bounds & (salary_numeric["salary_min"] > salary_numeric["salary_max"])
non_positive = (salary_numeric <= 0).any(axis=1) & any_bound

salary_quality = pd.DataFrame({
    "check": [
        "any numeric bound", "paired numeric bounds", "partial numeric bounds",
        "inverted paired ranges", "non-positive numeric values",
        "unparseable present salary_min", "unparseable present salary_max",
    ],
    "row_count": [
        int(any_bound.sum()), int(paired_bounds.sum()), int(partial_bounds.sum()),
        int(inverted_ranges.sum()), int(non_positive.sum()),
        int((salary_source_present["salary_min"] & ~lower_present).sum()),
        int((salary_source_present["salary_max"] & ~upper_present).sum()),
    ],
})
salary_quality["share_of_snapshot_pct"] = (salary_quality["row_count"] / len(jobs) * 100).round(2)
salary_quality["denominator_rows"] = len(jobs)
salary_quality

In [ ]:
currency_values = jobs["currency"].astype("string").fillna("<MISSING>")
currency_counts = currency_values.value_counts(dropna=False).rename_axis("currency").reset_index(name="count")
currency_counts["share_of_snapshot_pct"] = (currency_counts["count"] / len(jobs) * 100).round(2)
currency_counts["rows_with_any_numeric_bound"] = int(any_bound.sum())
currency_counts

## 8. Location, company, and text readiness

Raw location coverage and normalized-state coverage answer different questions. A populated `state` is not treated as validated geography. Company placeholders use a small, explicit heuristic and remain review candidates rather than automatic removals.

In [ ]:
COMPANY_PLACEHOLDERS = {
    "unknown", "confidential", "confidential company", "empresa confidencial",
    "n/a", "na", "not available", "sin nombre",
}
company_normalized = jobs["company"].astype("string").str.strip().str.lower()
company_present = field_present(jobs["company"])
company_placeholder = company_normalized.isin(COMPANY_PLACEHOLDERS).fillna(False)

readiness_checks = pd.DataFrame({
    "check": [
        "raw location present", "normalized state present (not accuracy-validated)",
        "company present", "company heuristic placeholder",
        "description present", "cleaned description present",
        "keywords payload present and non-empty", "embedding payload present and non-empty",
    ],
    "row_count": [
        int(field_present(jobs["raw_location"]).sum()),
        int(field_present(jobs["state"]).sum()),
        int(company_present.sum()),
        int(company_placeholder.sum()),
        int(field_present(jobs["description"]).sum()),
        int(field_present(jobs["cleaned_description"]).sum()),
        int(field_present(jobs["keywords_json"], serialized=True).sum()),
        int(field_present(jobs["embedding"], serialized=True).sum()),
    ],
})
readiness_checks["share_of_snapshot_pct"] = (readiness_checks["row_count"] / len(jobs) * 100).round(2)
readiness_checks["denominator_rows"] = len(jobs)
readiness_checks

## 9. Candidate analysis populations

These are mechanical eligibility gates, not validated study cohorts. Each definition is intentionally conservative and must be reviewed before a downstream notebook treats it as analytical truth.

In [ ]:
valid_salary_range = (
    paired_bounds
    & ~inverted_ranges
    & salary_numeric["salary_min"].gt(0)
    & salary_numeric["salary_max"].gt(0)
)
population_masks = {
    "duplicate study candidate": field_present(jobs["id"]) & field_present(jobs["job_url"]),
    "salary study candidate": valid_salary_range & field_present(jobs["currency"]),
    "demand study candidate": (
        field_present(jobs["title"])
        & analysis_jobs["date_scraped_parsed"].notna()
        & field_present(jobs["site"])
    ),
    "skill study candidate": (
        field_present(jobs["description"])
        & field_present(jobs["keywords_json"], serialized=True)
    ),
    "freshness study candidate": (
        analysis_jobs["date_scraped_parsed"].notna()
        & analysis_jobs["last_updated_parsed"].notna()
    ),
    "recommendation study candidate": (
        field_present(jobs["title"])
        & field_present(jobs["company"])
        & field_present(jobs["raw_location"])
        & field_present(jobs["description"])
    ),
}
population_definitions = {
    "duplicate study candidate": "non-empty id and job_url; no deduplication performed",
    "salary study candidate": "paired positive numeric bounds, ordered range, currency present",
    "demand study candidate": "title, parsed scrape date, and site present",
    "skill study candidate": "raw description and non-empty keywords payload present",
    "freshness study candidate": "parsed date_scraped and last_updated present",
    "recommendation study candidate": "title, company, raw location, and description present",
}
candidate_populations = pd.DataFrame([
    {
        "population": name,
        "mechanical_definition": population_definitions[name],
        "eligible_rows": int(mask.sum()),
        "share_of_snapshot_pct": round(mask.mean() * 100, 2),
        "denominator_rows": len(jobs),
        "validation_status": "CANDIDATE - REQUIRES REVIEW",
    }
    for name, mask in population_masks.items()
])
candidate_populations

## 10. Downstream decision table

The status vocabulary is fixed: **READY** means the mechanical prerequisite passes; **READY WITH LIMITATIONS** means a usable subset exists but interpretation is constrained; **BLOCKED** means the required evidence is absent. Reviewers should edit criteria in downstream work only after documenting the reason.

In [ ]:
def candidate_count(population: str) -> int:
    """Return the computed eligible-row count for one named population."""
    match = candidate_populations.loc[
        candidate_populations["population"].eq(population), "eligible_rows"
    ]
    return int(match.iloc[0])


keys_clean = (key_integrity[["empty_rows", "excess_duplicate_copies"]].to_numpy() == 0).all()
date_contract_matches = (
    scrape_dates.min() == EXPECTED_DATE_START
    and scrape_dates.max() == EXPECTED_DATE_END
    and int((analysis_jobs["date_scraped_parsed"] > SNAPSHOT_CUTOFF).sum()) == 0
)
skill_candidates = candidate_count("skill study candidate")
salary_candidates = candidate_count("salary study candidate")
recommendation_candidates = candidate_count("recommendation study candidate")

decision_table = pd.DataFrame([
    {
        "downstream_use": "Schema-based analysis",
        "criterion": "Exact checksum and dimensions; no missing or unexpected columns",
        "status": "READY" if not missing_columns and not unexpected_columns else "BLOCKED",
        "computed_evidence": f"shape={jobs.shape}; missing={len(missing_columns)}; unexpected={len(unexpected_columns)}",
    },
    {
        "downstream_use": "Duplicate study",
        "criterion": "ID and URL keys are non-empty; duplicates remain observable",
        "status": "READY" if keys_clean else "READY WITH LIMITATIONS",
        "computed_evidence": key_integrity.to_dict(orient="records"),
    },
    {
        "downstream_use": "Demand summaries within snapshot window",
        "criterion": "Fixed date contract passes; conclusions limited to Mar 11-Jul 27, 2026",
        "status": "READY WITH LIMITATIONS" if date_contract_matches else "BLOCKED",
        "computed_evidence": f"range={scrape_dates.min()} to {scrape_dates.max()}; missing dates={len(missing_calendar_dates)}",
    },
    {
        "downstream_use": "Annual or year-over-year claims",
        "criterion": "At least comparable annual periods are required",
        "status": "BLOCKED",
        "computed_evidence": "Current snapshot covers approximately 4.5 months",
    },
    {
        "downstream_use": "Salary study",
        "criterion": "At least one ordered, positive, paired numeric range with currency",
        "status": "READY WITH LIMITATIONS" if salary_candidates > 0 else "BLOCKED",
        "computed_evidence": f"candidate rows={salary_candidates:,} of {len(jobs):,}; periods and normalization unverified",
    },
    {
        "downstream_use": "Normalized-state geography",
        "criterion": "State must be populated and accuracy-validated for the intended geography",
        "status": "BLOCKED",
        "computed_evidence": f"state present={int(field_present(jobs['state']).sum()):,} of {len(jobs):,}; accuracy not audited",
    },
    {
        "downstream_use": "Skill study",
        "criterion": "Descriptions and non-empty NLP fields exist for a reviewable subset",
        "status": "READY WITH LIMITATIONS" if skill_candidates > 0 else "BLOCKED",
        "computed_evidence": f"candidate rows={skill_candidates:,}; cleaned text and embeddings audited separately",
    },
    {
        "downstream_use": "Listing survival or closure analysis",
        "criterion": "Repeated observations or explicit closure history are required",
        "status": "BLOCKED",
        "computed_evidence": "A current-row snapshot cannot reconstruct listing survival",
    },
    {
        "downstream_use": "Recommendation baseline",
        "criterion": "Core listing text and context exist for a reviewable subset",
        "status": "READY WITH LIMITATIONS" if recommendation_candidates > 0 else "BLOCKED",
        "computed_evidence": f"candidate rows={recommendation_candidates:,} of {len(jobs):,}; relevance not validated",
    },
])
decision_table

## Next step

Review the code, definitions, denominators, and decision criteria first. Execute this notebook only after an explicit request; then review the computed evidence before creating any downstream market metric.